# 05 -- Bayesian TDoA Integration (Kalman Filter)

For a **stationary target**, repeated TDoA measurements from the same set of
receivers are independent noisy observations of the same underlying delay
vector. A linear Kalman filter provides the optimal way to fuse these
measurements: with each update the posterior variance shrinks, converging
toward the Cramer-Rao Lower Bound (CRLB). This notebook demonstrates the
integration loop, tracks variance reduction, and shows how the
`ConvergenceMonitor` detects when further observations yield negligible
improvement.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pingu.integrator.kalman import TDoAKalmanFilter
from pingu.integrator.convergence import ConvergenceMonitor
from pingu.tdoa.uncertainty import crlb_tdoa

# Reproducible random state
rng = np.random.default_rng(42)

# Number of receiver pairs: C(5, 2) = 10
N_PAIRS = 10

print(f"Receiver pairs: {N_PAIRS}")

## Kalman Filter Setup

The Kalman filter state vector holds the 10 TDoA values (one per receiver pair).
Because the target is stationary, the state-transition matrix is the **identity**
-- the delays do not change between time steps. A tiny process noise
(`1e-12 s^2`) is included to keep the filter from becoming numerically singular,
while the initial covariance diagonal is set to `1e-6 s^2` (a diffuse prior that
lets the first few measurements dominate).

In [ ]:
# Create the Kalman filter
kf = TDoAKalmanFilter(
    n_pairs=N_PAIRS,
    process_noise=1e-12,
    initial_variance=1e-6,
)

# Inspect initial state
state0 = kf.get_state()
print("Initial state vector (delays):")
print(f"  {state0.delays}")
print(f"\nInitial covariance diagonal (s^2):")
print(f"  {np.diag(state0.covariance)}")
print(f"\nNumber of updates applied: {kf.n_updates}")

## Feeding Noisy Measurements

We define a true TDoA vector (10 values spread linearly from -100 us to
+100 us) and generate 200 noisy measurement vectors by adding Gaussian noise
with a standard deviation of 0.1 us (variance = 1e-14 s^2). The
`predict` + `update` loop fuses each measurement into the posterior.

In [ ]:
# True TDoA vector: 10 values from -100 us to +100 us
true_tdoas = np.linspace(-1e-4, 1e-4, N_PAIRS)  # seconds

# Measurement noise parameters
MEAS_STD = 1e-7  # 0.1 us standard deviation
MEAS_VAR = MEAS_STD ** 2  # 1e-14 s^2
N_UPDATES = 200

# Storage for tracking filter evolution
state_history = np.zeros((N_UPDATES, N_PAIRS))   # filtered TDoA at each step
var_history = np.zeros((N_UPDATES, N_PAIRS))      # diagonal variance at each step
measurements_all = np.zeros((N_UPDATES, N_PAIRS)) # raw measurements

# Reset the filter to ensure a clean start
kf.reset()

for k in range(N_UPDATES):
    # Generate noisy measurement
    noise = rng.normal(0.0, MEAS_STD, size=N_PAIRS)
    z = true_tdoas + noise
    measurements_all[k] = z

    # Kalman predict + update
    kf.predict()
    kf.update(
        measurements=z,
        measurement_noise=np.full(N_PAIRS, MEAS_VAR),
        timestamp=float(k),
    )

    # Record state
    st = kf.get_state()
    state_history[k] = st.delays
    var_history[k] = np.diag(st.covariance)

print(f"Completed {N_UPDATES} predict+update cycles.")
print(f"Final state (pair 0): {state_history[-1, 0]*1e6:.4f} us (true: {true_tdoas[0]*1e6:.4f} us)")
print(f"Final variance (pair 0): {var_history[-1, 0]:.4e} s^2")

In [ ]:
# Plot filtered TDoA for pair 0 vs measurement number
pair_idx = 0
updates = np.arange(1, N_UPDATES + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(
    updates, measurements_all[:, pair_idx] * 1e6,
    s=4, alpha=0.3, color="gray", label="Raw measurements",
)
ax.plot(
    updates, state_history[:, pair_idx] * 1e6,
    color="tab:blue", linewidth=1.5, label="Kalman filtered estimate",
)
ax.axhline(
    true_tdoas[pair_idx] * 1e6,
    color="red", linestyle="--", linewidth=1.0, label="True value",
)
ax.set_xlabel("Update number")
ax.set_ylabel("TDoA (us)")
ax.set_title(f"Kalman Filter Convergence -- Pair {pair_idx}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The filtered estimate (blue line) converges rapidly to the true value (red dashed).")
print("Raw measurements (gray dots) have much higher scatter.")

## Variance Reduction Over Time

The posterior variance should decrease monotonically with each update,
approaching the theoretical CRLB. For identical independent measurements
the variance decreases as `~1/N`, where N is the number of updates.

In [ ]:
# Compute a representative CRLB for comparison
# Using typical HF parameters: bandwidth=6 kHz, SNR=20 dB, integration_time = N_SAMPLES/FS
BANDWIDTH_HZ = 6000.0
SNR_LINEAR = 10.0 ** (20.0 / 10.0)  # 20 dB
INTEGRATION_TIME = 4096.0 / 48000.0  # ~85 ms
crlb_single = crlb_tdoa(
    bandwidth=BANDWIDTH_HZ,
    snr_linear=SNR_LINEAR,
    integration_time=INTEGRATION_TIME,
)

# Theoretical 1/N line: initial variance / N
initial_var = var_history[0, 0]
theoretical_1_over_n = MEAS_VAR / updates  # for large N, Kalman var -> R/N

fig, ax = plt.subplots(figsize=(10, 5))
for p in range(N_PAIRS):
    ax.semilogy(updates, var_history[:, p], alpha=0.5, linewidth=0.8)
ax.semilogy(
    updates, theoretical_1_over_n,
    color="black", linestyle=":", linewidth=1.5, label="Theoretical 1/N",
)
ax.axhline(
    crlb_single, color="red", linestyle="--", linewidth=1.5,
    label=f"CRLB = {crlb_single:.2e} s^2",
)
ax.set_xlabel("Update number")
ax.set_ylabel("Posterior variance (s^2)")
ax.set_title("Variance Reduction Over Time (all 10 pairs)")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("Variance decreases monotonically, tracking the theoretical 1/N curve.")
print(f"CRLB (single observation): {crlb_single:.4e} s^2")

In [ ]:
# Print improvement ratios at key milestones
for n_check in [50, 100, 200]:
    idx = n_check - 1  # zero-indexed
    initial_mean_var = np.mean(var_history[0, :])
    current_mean_var = np.mean(var_history[idx, :])
    ratio = initial_mean_var / current_mean_var
    print(f"After {n_check:>3d} updates: mean variance = {current_mean_var:.4e} s^2, "
          f"improvement ratio = {ratio:.1f}x")

# Verify >10x improvement at 100 updates
ratio_100 = np.mean(var_history[0, :]) / np.mean(var_history[99, :])
print(f"\nVerification: improvement at 100 updates = {ratio_100:.1f}x "
      f"({'PASS: > 10x' if ratio_100 > 10 else 'BELOW 10x'})")

## Convergence Monitor

The `ConvergenceMonitor` wraps the variance-tracking logic and provides a
`is_converged()` predicate that fires when the posterior variance has dropped
sufficiently close to the CRLB (or when the rate of improvement stalls).

In [ ]:
# Re-run the Kalman loop with the ConvergenceMonitor attached
kf2 = TDoAKalmanFilter(
    n_pairs=N_PAIRS,
    process_noise=1e-12,
    initial_variance=1e-6,
)
monitor = ConvergenceMonitor(n_pairs=N_PAIRS)

converged_at = None  # update number where is_converged() first returns True
rng2 = np.random.default_rng(42)  # same seed for reproducibility

for k in range(N_UPDATES):
    noise = rng2.normal(0.0, MEAS_STD, size=N_PAIRS)
    z = true_tdoas + noise

    kf2.predict()
    kf2.update(
        measurements=z,
        measurement_noise=np.full(N_PAIRS, MEAS_VAR),
        timestamp=float(k),
    )

    # Feed covariance to the monitor (with CRLB as diagonal)
    st = kf2.get_state()
    crlb_diag = np.full(N_PAIRS, crlb_single)
    monitor.update(st.covariance, crlb=np.diag(crlb_diag))

    if converged_at is None and monitor.is_converged():
        converged_at = k + 1  # 1-indexed

if converged_at is not None:
    print(f"Convergence detected at update: {converged_at}")
else:
    print("Filter did not converge within the allotted updates.")

print(f"Final improvement ratio: {monitor.get_improvement_ratio():.1f}x")

# Plot variance history from the monitor
var_hist = monitor.get_variance_history()  # shape (n_updates, n_pairs)

fig, ax = plt.subplots(figsize=(10, 5))
for p in range(N_PAIRS):
    ax.semilogy(np.arange(1, len(var_hist) + 1), var_hist[:, p], alpha=0.5, linewidth=0.8)
ax.axhline(crlb_single, color="red", linestyle="--", linewidth=1.5, label=f"CRLB = {crlb_single:.2e} s^2")
if converged_at is not None:
    ax.axvline(converged_at, color="green", linestyle="-.", linewidth=1.5, label=f"Converged at update {converged_at}")
ax.set_xlabel("Update number")
ax.set_ylabel("Posterior variance (s^2)")
ax.set_title("Convergence Monitor -- Variance History")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## Summary

- The **Kalman filter** reduces TDoA variance approximately as **1/N** (where N
  is the number of measurement updates), which is the optimal rate for fusing
  independent identically-distributed observations of a static quantity.
- The **convergence monitor** automatically detects when the posterior variance
  has dropped below a configurable fraction of the CRLB, signalling that the
  integration loop can stop. This avoids wasting computation on diminishing
  returns.